<a href="https://colab.research.google.com/github/hosseinta2/LLM-from-scratch/blob/main/instruction_fine_tuning_with_LLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os
import urllib
def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()
    with open(file_path, "r") as file:
        data = json.load(file)
    return data
file_path = "instruction-data.json"
url = (
"https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
"/main/ch07/01_main-chapter-code/instruction-data.json" )
data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


In [2]:
print(data[200])

{'instruction': 'What is the chemical formula for ammonia?', 'input': '', 'output': 'The chemical formula for ammonia is NH3.'}


In [3]:
def format_input(entry):
    instruction_text = (f"Below is an instruction that describes a task. "
    f"Write a response that appropriately completes the request."
                        f"\n\n### Instruction:\n{entry['instruction']}"
          )
    input_text = (f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    response_text = f"\n\n### Response:\n{entry['output']}"
    return instruction_text + input_text + response_text

In [4]:
model_input = format_input(data[10])
print(model_input)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is the contraction for "will not"?

### Response:
The contraction for "will not" is "won't".


In [11]:
train_data = data[:880]
test_data = data[880:]
import torch
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
padding_id = 50256

def padded(input, padding_id=50256, ignore_index = -100, allowed_max_length = 1024):
  input_padded = []
  target_padded = []
  max_length = max(len(x)+1 for x in input)
  if max_length>allowed_max_length:
    max_length = allowed_max_length
  for x in input:
    padded_x = x+(max_length-len(x))*[padding_id]
    padded_x2 = x+[padding_id]+(max_length-len(x)-1)*[ignore_index]
    input_padded.append(padded_x[:-1])
    target_padded.append(padded_x2[1:])
  return torch.tensor(input_padded),torch.tensor(target_padded)

x,y = padded([[1,2,3],[4,5],[6]],padding_id)
print(y.shape)

torch.Size([3, 3])


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.backends.mps.is_available():
  device = torch.device("mps")
print(device)

cuda


In [7]:
train_data_list = []
test_data_list = []
for instruction in train_data:
  text = format_input(instruction)
  tokenized_text = tokenizer.encode(text)
  train_data_list.append(tokenized_text)
for instruction in test_data:
  text = format_input(instruction)
  tokenized_text = tokenizer.encode(text)
  test_data_list.append(tokenized_text)
x_tr,y_tr = padded(train_data_list)
x_te,y_te = padded(test_data_list)


In [8]:
from torch.utils.data import Dataset, DataLoader,TensorDataset

batched_train_data = TensorDataset(x_tr,y_tr)
batched_test_data = TensorDataset(x_te,y_te)

training_batches = DataLoader(batched_train_data,batch_size=8,shuffle=True,drop_last=True)
test_batches = DataLoader(batched_test_data,batch_size=8,shuffle=False,drop_last=False)



In [10]:
import urllib.request

url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch05/"
    "01_main-chapter-code/gpt_download.py"
)
filename = url.split('/')[-1]
urllib.request.urlretrieve(url, filename)

from gpt_download import download_and_load_gpt2 #importing GPT2 OAI weights
from model_and_load_weight import GPT,load_weights_into_gpt #importing the model and the function to load gpt2 OAI weight



BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True
}
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
BASE_CONFIG.update(model_configs["gpt2-medium (355M)"])

model_size = "355M"

settings, params = download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2"
)
model = GPT(BASE_CONFIG)
load_weights_into_gpt(model, params)

checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 144kiB/s]
encoder.json: 100%|██████████| 1.04M/1.04M [00:00<00:00, 2.59MiB/s]
hparams.json: 100%|██████████| 90.0/90.0 [00:00<00:00, 72.0kiB/s]
model.ckpt.data-00000-of-00001: 100%|██████████| 498M/498M [00:46<00:00, 10.6MiB/s]
model.ckpt.index: 100%|██████████| 5.21k/5.21k [00:00<00:00, 12.9MiB/s]
model.ckpt.meta: 100%|██████████| 471k/471k [00:00<00:00, 1.57MiB/s]
vocab.bpe: 100%|██████████| 456k/456k [00:00<00:00, 1.42MiB/s]
checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 180kiB/s]
encoder.json: 100%|██████████| 1.04M/1.04M [00:00<00:00, 2.14MiB/s]
hparams.json: 100%|██████████| 91.0/91.0 [00:00<00:00, 189kiB/s]
model.ckpt.data-00000-of-00001: 100%|██████████| 1.42G/1.42G [01:38<00:00, 14.5MiB/s]
model.ckpt.index: 100%|██████████| 10.4k/10.4k [00:00<00:00, 18.8MiB/s]
model.ckpt.meta: 100%|██████████| 927k/927k [00:00<00:00, 3.04MiB/s]
vocab.bpe: 100%|██████████| 456k/456k [00:00<00:00, 1.80MiB/s]


In [12]:
def format_input_only(entry):
    instruction_text = (f"Below is an instruction that describes a task. "
    f"Write a response that appropriately completes the request."
                        f"\n\n### Instruction:\n{entry['instruction']}"
          )
    input_text = (f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    return instruction_text + input_text
def generate_text(model,tokenized_text,max_new_context_size,context_size,temp,topk): # generating max_new_tokens for a text and a model with context_size,
                                                                           #used for testing the model's output during training
  for _ in range(max_new_context_size):
      ids = tokenized_text[:,-context_size:]
      logits = model(ids) #batch_size*context_size*vocab_size
      last_logit = logits[:,-1,:] #batch_size*vocab_size
      top_val,top_pos = torch.topk(last_logit,topk)
      last_logit = torch.where(condition = last_logit<top_val.squeeze(0)[-1],
                               input = torch.tensor(float('-inf')).to(last_logit.device),
                               other = last_logit)

      scores = torch.softmax(last_logit/temp,dim=-1) #batch_size*vocab_size
      #max_id = torch.argmax(scores,dim=-1, keepdim=True) #batch_size*1
      new_id = torch.multinomial(scores,num_samples=1)
      tokenized_text = torch.cat((tokenized_text,new_id),dim=-1)#cat(batch_size*len_context,batch_size*1)=batch_size * (len_context+1)
      if new_id==50256:
        break

  return tokenized_text

In [ ]:
import time
model = GPT(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.to(device)
loss_func = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=7e-5,weight_decay=0.1)


def batch_loss(input,target,model,device):# cross_entropy loss for a batch of data
  input = input.to(device)
  target = target.to(device)
  output = model(input)
  output = output.flatten(0,1)
  target = target.flatten()
  loss_val = loss_func(output,target)
  return loss_val

train_loss,eval_loss =[],[]
steps = -1
freq = 50
num_epochs = 5

model.train()

start_time = time.time()
for epoch in range(num_epochs):
  for x,y in training_batches:
    optimizer.zero_grad()
    los = batch_loss(x,y,model,device)
    los.backward()
    optimizer.step()
    steps+=1
    if steps%freq==0:
      model.eval()
      train_loss_value = 0
      val_loss_value = 0
      n_batches = 0
      with torch.no_grad():
        for train_x,train_y in training_batches:
          train_loss_value += batch_loss(train_x,train_y,model,device).item()
          n_batches +=1
          if n_batches>10:
            break
        train_loss_value  =  train_loss_value/n_batches
        n_batches = 0
        for val_x,val_y in test_batches:
          val_loss_value += batch_loss(val_x,val_y,model,device).item()
          n_batches +=1
          if n_batches>10:
            break
        val_loss_value = val_loss_value/n_batches
        sample_input = test_data[0]
        sample_input =format_input_only(sample_input)
        sample_padded,target_padded = padded([tokenizer.encode(sample_input)])
        sample_padded =sample_padded.to(device)
        sample_out = generate_text(model,sample_padded,5,20,1,3)
        sample_out_text = tokenizer.decode(list(sample_out.squeeze()))
      train_loss.append(train_loss_value)
      eval_loss.append(val_loss_value)
      print("Epoch = ", epoch, "train_loss = ", train_loss_value, "val_loss = ", val_loss_value,"\n",sample_out_text,"\n")
      model.train()
end_time = time.time()



print("total time = ", (end_time-start_time)/60)


Epoch =  0 train_loss =  3.376423705707897 val_loss =  3.3773506337946113 
 Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Rewrite the sentence using alliteration.

### Input:
The wind blew softly.

The wind blew 

Epoch =  0 train_loss =  0.9904089570045471 val_loss =  1.0431831208142368 
 Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Rewrite the sentence using alliteration.

### Input:
The wind blew softly.

The wind blew 



In [ ]:
import matplotlib.pyplot as plt
x_axis = [i*num_epochs/len(train_loss) for i in range(len(train_loss))]
fig,ax = plt.subplots(figsize = (5,3))
ax.plot(x_axis,train_loss, label=f"Train")
ax.plot(x_axis, eval_loss,label =f"Validation")
ax.legend()
ax.set_xlabel("epochs")
ax.set_ylabel("Loss")
plt.show()

In [ ]:
torch.save(model.state_dict(),"instruction_fine_tuning.pth")

In [ ]:
model.eval()
sample_input = test_data[5]
sample_input =format_input_only(sample_input)
sample_padded,target_padded = padded([tokenizer.encode(sample_input)])
sample_padded =sample_padded.to(device)
sample_out = generate_text(model,sample_padded,30,40,.5,3)
sample_out_text = tokenizer.decode(list(sample_out.squeeze()))
print(sample_out_text)